In [0]:
spark.sql("DROP TABLE IF EXISTS de_mini_project.silver.inventory")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS de_mini_project.silver.inventory (
    sku_id STRING,
    store_id STRING,
    stocks_qty INTEGER,
    last_audit_date DATE   
)
""")

In [0]:
spark.sql("""
INSERT INTO de_mini_project.silver.inventory (sku_id, store_id, stocks_qty, last_audit_date)
WITH inventory_cleaned AS (
    SELECT 
        _sku_id_ AS sku_id,
        st_id_ref AS store_id,
        CAST(_stock_on_hand_ AS INTEGER) AS stocks_qty,
        try_to_date(CAST(last_audit_dt AS STRING), 'yyyyMMdd') AS last_audit_date
    FROM de_mini_project.azure_blob_storage.inventory
)
SELECT 
    sku_id, 
    store_id, 
    stocks_qty, 
    last_audit_date
FROM inventory_cleaned
QUALIFY ROW_NUMBER() OVER (PARTITION BY sku_id, store_id ORDER BY last_audit_date DESC) = 1
""")